In [ ]:
import picsure

In [ ]:
my_token = ""

with open("token.txt") as f:
    my_token = f.read().strip()

auth_hpds_session = picsure.connect(
    picsure.Platform.BDC_DEV_AUTHORIZED,
    token=my_token,
    dev_mode=True
)

In [ ]:
facets = auth_hpds_session.facets()
facets.add("dataset_id", "phs000810")
facets.add("dataset_id", "phs000007")
facets.view()

In [ ]:
results = auth_hpds_session.searchDictionary("age", facets=facets)
results

In [ ]:
age5_phs000007 = results[results["display"] == "age5"]
age5_phs000007 = age5_phs000007[age5_phs000007["name"] == "phv00177938"]
age5_phs000007_clause = picsure.createSubQuery(
    age5_phs000007["conceptPath"].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    min=30,
    max=40
)

In [ ]:
fhs_facet = auth_hpds_session.facets()
fhs_facet.add("dataset_id", "phs000007")
fhs_sex_results = auth_hpds_session.searchDictionary("phv00253990", facets=fhs_facet)
fhs_sex_results

In [ ]:
fhs_sex_male_clause = picsure.createSubQuery(
    fhs_sex_results['conceptPath'].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    ["Male"]
)

fhsMaleAnd30To40 = picsure.buildQuery(
    [fhs_sex_male_clause, age5_phs000007_clause],
    operator=picsure.GroupOperator.AND
)

fhs_sex_female_clause = picsure.createSubQuery(
    fhs_sex_results['conceptPath'].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    ["Female"]
)

fhsFemaleAnd30To40 = picsure.buildQuery(
    [fhs_sex_female_clause, age5_phs000007_clause],
    operator=picsure.GroupOperator.AND
)

fhs_FemaleAges30to40_OR_MaleAges30to40 = picsure.buildQuery(
    [fhsFemaleAnd30To40, fhsMaleAnd30To40],
    operator=picsure.GroupOperator.OR
)

fhs_FemaleAges30to40_OR_MaleAges30to40.to_query_json()

In [ ]:
auth_hpds_session.runQuery(fhs_FemaleAges30to40_OR_MaleAges30to40)
# Results are within expected margin. Verified results with UI. UI Result: 77±3

In [ ]:
auth_hpds_session.saveQueryByName(fhs_FemaleAges30to40_OR_MaleAges30to40, "Created from API adapters - fhs_FemaleAges30to40_OR_MaleAges30to40", overwrite=True)

In [ ]:
query = auth_hpds_session.loadQueryByID("051e5f8e-53a7-4d7d-b199-e0890b9afe05")

In [ ]:
auth_hpds_session.runQuery(query)